In [1]:
import os

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import AuthorRole, ChatMessageContent
from semantic_kernel.functions import KernelFunctionFromPrompt

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
def _create_kernel_with_chat_completion() -> Kernel:
    kernel = Kernel()

    client = AsyncOpenAI(
        api_key=os.environ.get("API_KEY"),
        base_url=os.environ.get("API_URL"),
    )

    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id=os.environ.get("MODEL_FREE_8B"),
            async_client=client,
        )
    )

    return kernel

In [3]:
async def main():
    REVIEWER_NAME = "Concierge"
    REVIEWER_INSTRUCTIONS = """
    你是一位酒店礼宾员，对为旅行者提供最本地化和最真实的体验有自己的见解。
    目标是确定前台旅行代理是否为旅行者推荐了最佳的非旅游体验。
    如果是，请说明已批准。
    如果不是，请提供关于如何完善建议的见解，但不要使用具体示例。
    """
    agent_reviewer = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion(),
        name=REVIEWER_NAME,
        instructions=REVIEWER_INSTRUCTIONS,
    )

    FRONTDESK_NAME = "FrontDesk"
    FRONTDESK_INSTRUCTIONS = """
    你是一位有十年经验的前台旅行代理，以简洁著称，因为你需要处理许多客户。
    目标是为旅行者提供最佳的活动和地点推荐。
    每次回复只提供一个推荐。
    你专注于手头的目标。
    不要浪费时间闲聊。
    在完善想法时考虑建议。
    """
    agent_writer = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion(),
        name=FRONTDESK_NAME,
        instructions=FRONTDESK_INSTRUCTIONS,
    )

    termination_function = KernelFunctionFromPrompt(
        function_name="termination",
        prompt="""
        确定推荐流程是否完成。
        
        当礼宾员对前台提出的任何建议表示批准时，流程即完成。
        寻找诸如"approved"（已批准）、"this recommendation is approved"（此建议已批准）或任何明确表示礼宾员对建议满意的短语。
        
        如果礼宾员在最近的回复中给出了批准，请回复：yes
        否则，请回复：no
        
        历史记录：
        {{$history}}
        """,
    )

    selection_function = KernelFunctionFromPrompt(
        function_name="selection",
        prompt=f"""
        根据最近的参与者确定对话中下一个发言的参与者。
        仅说明下一个发言的参与者的名称。
        没有参与者应该连续发言超过一次。
        
        只能从以下参与者中选择：
        - {REVIEWER_NAME}
        - {FRONTDESK_NAME}
        
        选择下一个参与者时始终遵循这些规则，每次对话至少进行4轮：
        - 用户输入后，轮到 {FRONTDESK_NAME}。
        - {FRONTDESK_NAME} 回复后，轮到 {REVIEWER_NAME}。
        - {REVIEWER_NAME} 提供反馈后，轮到 {FRONTDESK_NAME}。

        历史记录：
        {{{{$history}}}}
        """,
    )

    chat = AgentGroupChat(
        agents=[agent_writer, agent_reviewer],
        termination_strategy=KernelFunctionTerminationStrategy(
            agents=[agent_reviewer],
            function=termination_function,
            kernel=_create_kernel_with_chat_completion(),
            result_parser=lambda result: str(result.value[0]).lower() == "yes",
            history_variable_name="history",
            maximum_iterations=10,
        ),
        selection_strategy=KernelFunctionSelectionStrategy(
            function=selection_function,
            kernel=_create_kernel_with_chat_completion(),
            result_parser=lambda result: str(
                result.value[0]) if result.value is not None else FRONTDESK_NAME,
            agent_variable_name="agents",
            history_variable_name="history",
        ),
    )

    user_input = "I would like to go to Paris."

    await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))
    print(f"# User: '{user_input}'")

    async for content in chat.invoke():
        print(f"# Agent - {content.name or '*'}: '{content.content}'")

    print(f"# IS COMPLETE: {chat.is_complete}")

await main()

Function failed. Error: Argument 'history' has a value that doesn't support automatic encoding. Set allow_dangerously_set_content to 'True' for this argument and implement custom encoding, or provide the value as a string.
Kernel Function Selection Strategy next method failed
Traceback (most recent call last):
  File "/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/semantic_kernel/agents/strategies/selection/kernel_function_selection_strategy.py", line 95, in select_agent
    result = await self.function.invoke(kernel=self.kernel, arguments=arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/semantic_kernel/functions/kernel_function.py", line 293, in invoke
    raise e
  File "/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/semantic_kernel/functions/kernel_function.py", line 278, in invoke
    await sta

# User: 'I would like to go to Paris.'


AgentChatException: Failed to select agent